# Lecture 23 · Transplant Surgery: the notebook

*The Anatomy of AI.* In the lecture we ran GPT-2 small on two sentences one word apart and moved numbers from one run into the other to find where its decision is made; then we found the heads that make it, and where a fact is stored. Here you do every transplant yourself:

1. **GPT-2 small, with hooks** (Lecture 19's model, opened up so any number can be read or replaced).
2. **Two sentences, one number** (Plates I, III).
3. **The 195 transplants** and the map they make (Plates IV–VIII).
4. **144 heads, one at a time** (Plate IX).
5. **Who writes the answer, and what they look at** (Plates X–XII, XIV).
6. **A transplant along one wire**: path patching (Plate XIII).
7. **Self-repair** (Plate XVI).
8. **Where a fact lives**: causal tracing (Plates XVII–XX).

Every key number is checked against the lecture's own logs with `check()`. It runs on a free Colab **T4 GPU** in about 10 minutes. No keys needed.

**Naming.** As on the slides, blocks and heads count from 1: head 10.10 is block 10, head 10 (Wang et al. call it 9.9). The code counts from 0, so `hn(9, 9)` prints "10.10".

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3", "regex==2024.11.6")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path
T0_START = time.time()

The lecture's own numbers, from its logs (`lectures/23-transplant-surgery/experiments/raw/`).

In [ ]:
LECTURE = {
 "tokens": [
  "<|endoftext|>",
  "When",
  " Mary",
  " and",
  " John",
  " went",
  " to",
  " the",
  " store",
  ",",
  " John",
  " gave",
  " a",
  " drink",
  " to"
 ],
 "top5_clean": [
  [
   " Mary",
   0.677
  ],
  [
   " them",
   0.1283
  ],
  [
   " the",
   0.0476
  ],
  [
   " John",
   0.0235
  ],
  [
   " her",
   0.0141
  ]
 ],
 "top5_corrupt": [
  [
   " John",
   0.4114
  ],
  [
   " them",
   0.2044
  ],
  [
   " the",
   0.1191
  ],
  [
   " Mary",
   0.0345
  ],
  [
   " her",
   0.0248
  ]
 ],
 "ld_clean": 3.3616,
 "ld_corrupt": -2.4787,
 "john_col": [
  1.0,
  0.9983,
  0.9994,
  0.9923,
  0.9775,
  0.9294,
  0.9307,
  0.8745,
  0.6294,
  0.0893,
  0.0264,
  0.0163,
  0.0
 ],
 "to_col": [
  0.0,
  0.0007,
  0.0027,
  0.0025,
  0.0018,
  -0.0019,
  -0.0011,
  0.037,
  0.331,
  0.97,
  1.0217,
  0.9963,
  1.0
 ],
 "dist_end": [
  0.0,
  0.55,
  0.626,
  0.85,
  1.254,
  1.558,
  2.272,
  5.9,
  18.119,
  39.092,
  71.677,
  67.423,
  67.596
 ],
 "head_pair_9_11": 0.2527,
 "head_pair_10_10": 0.2448,
 "head_pair_11_8": -0.4606,
 "noise_to_9": 0.1247,
 "noise_john_8": 0.3762,
 "jobs_top_clean": [
  " Apple",
  0.8361
 ],
 "jobs_top_noised": [
  " the",
  0.3506
 ],
 "sigma": 0.1352,
 "noise_length": 11.288,
 "emb_length": [
  3.551,
  3.979
 ],
 "n_prompts": 192,
 "ld_mean_clean": 3.7294,
 "ld_mean_abc": 0.1269,
 "top1_io": 0.9635,
 "head192_12_3": -0.8018,
 "dla_10_10": 3.1885,
 "dla_10_7": 1.0822,
 "dla_11_8": -2.1416,
 "name_movers": [
  "9.9",
  "9.6",
  "10.0"
 ],
 "att_end_io_10_10": 0.7993,
 "att_end_s2_9_7": 0.8133,
 "att_s2_s1_4_1": 0.7783,
 "att_induction_6_6": 0.8463,
 "path_q_9_7": -0.1293,
 "s_inhibition": [
  "8.6",
  "7.9",
  "8.10",
  "7.3"
 ],
 "path_v_6_6": -0.2376,
 "ko_after": 3.6165,
 "ko_direct": 4.8768,
 "ko_change_11_8": 2.3039,
 "n_known": 27,
 "p_clean_mean": 0.3782,
 "p_corrupt_mean": 0.0124,
 "jobs_resid_after_4": 0.5247,
 "mlp_last_subject": [
  0.0986,
  0.0902,
  0.0026
 ],
 "attn_last_token_9_10": 0.1475,
 "eiffel_best": "The Eiffel Tower stands in the city of",
 "eiffel_p": 0.096
}

## Part 1 · GPT-2 small, with hooks

The tokenizer and the weight file are exactly Lecture 19's. The forward pass is too, with one addition: a **hook** at every place we may want to look or operate. A hook is a name, such as `z.9` (what each head of block 10 has read, before it is written back), and a function that receives those numbers and may return different ones. TransformerLens, which the lecture used, works the same way; ours is about 40 lines.

In [ ]:
# GPT-2 small, exactly as Lecture 19 built it: no library model, only its files from Hugging Face (no key needed).
import regex as re
HF = "https://huggingface.co/openai-community/gpt2/resolve/main/"
fetch(HF + "vocab.json", "vocab.json", "GPT-2's token list")
fetch(HF + "merges.txt", "merges.txt", "GPT-2's 50,000 merges")
fetch(HF + "model.safetensors", "gpt2.safetensors", "GPT-2 small's weights")

# --- the tokenizer: bytes -> printable characters, a regex split, then the merges in rank order (Lecture 19, Plate VI)
VOCAB = json.load(open("vocab.json", encoding="utf-8"))
ID2TOK = {i: t for t, i in VOCAB.items()}
RANK = {tuple(m.split()): r for r, m in enumerate(l for l in open("merges.txt", encoding="utf-8").read().split("\n")[1:] if l)}
_bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
_cs = _bs + [256 + n for n in range(256 - len(_bs))]
_bs = _bs + [b for b in range(256) if b not in _bs]
B2U = dict(zip(_bs, map(chr, _cs))); U2B = {u: b for b, u in B2U.items()}
SPLIT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
_seen = {}

def bpe(piece):
    if piece not in _seen:
        w = list(piece)
        while len(w) > 1:
            r, i = min((RANK.get(p, 1 << 30), i) for i, p in enumerate(zip(w, w[1:])))
            if r == 1 << 30:
                break
            a, b, out, j = w[i], w[i + 1], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    out.append(a + b); j += 2
                else:
                    out.append(w[j]); j += 1
            w = out
        _seen[piece] = w
    return _seen[piece]

def encode(text):
    return [VOCAB[t] for m in SPLIT.findall(text) for t in bpe("".join(B2U[b] for b in m.encode("utf-8")))]

def decode(ids):
    return bytes(U2B[ch] for i in ids for ch in ID2TOK[int(i)]).decode("utf-8", errors="replace")

# --- the weights: a safetensors file is an 8-byte length, a JSON table of contents, then the raw numbers
def read_safetensors(path):
    raw = open(path, "rb").read()
    n = int.from_bytes(raw[:8], "little")
    head = json.loads(raw[8:8 + n])
    out = {}
    for name, h in head.items():
        if name != "__metadata__" and not name.endswith((".attn.bias", ".attn.masked_bias")):   # masks, not weights
            a, b = h["data_offsets"]
            out[name] = torch.from_numpy(np.frombuffer(raw, dtype=np.float32, count=(b - a) // 4, offset=8 + n + a)
                                         .reshape(h["shape"]).copy()).to(DEV)
    return out

SD = read_safetensors("gpt2.safetensors")
WTE, WPE = SD["wte.weight"], SD["wpe.weight"]
print(f"GPT-2 small: {sum(v.numel() for v in SD.values()):,} weights")


NAMES_OF = {}                                       # hook name -> what it holds

@torch.no_grad()
def gpt2(ids, hooks=None, keep=(), last_only=False):
    """ids: [B, T] token ids. hooks: {name: fn(numbers) -> numbers}. keep: hook names to record ("all" for every one).
    Returns the scores [B, T, 50257] (only the last position's, [B, 1, 50257], with last_only) and what was kept."""
    ids = torch.as_tensor(ids, device=DEV)
    B, T = ids.shape
    cache = {}
    def H(name, x):
        if hooks and name in hooks:
            x = hooks[name](x)
        if keep == "all" or name in keep:
            cache[name] = x
        return x
    x = H("embed", WTE[ids]) + WPE[:T]                                    # the token's arrow (+ its position's)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=DEV))
    for l in range(12):
        p = lambda n: SD[f"h.{l}.{n}"]
        x = H(f"resid_pre.{l}", x)                                        # the running total before block l+1
        h = layer_norm(x, p("ln_1.weight"), p("ln_1.bias"))
        q, k, v = (h @ p("attn.c_attn.weight") + p("attn.c_attn.bias")).split(768, -1)
        q, k, v = (H(f"{n}.{l}", t.view(B, T, 12, 64)) for n, t in (("q", q), ("k", k), ("v", v)))
        s = torch.einsum("bqhd,bkhd->bhqk", q, k) / 8
        a = H(f"pattern.{l}", s.masked_fill(~mask, float("-inf")).softmax(-1))   # [B, head, from, to]
        z = H(f"z.{l}", torch.einsum("bhqk,bkhd->bqhd", a, v))                   # [B, T, head, 64]
        att = H(f"attn_out.{l}", torch.einsum("bqhd,hdm->bqm", z, p("attn.c_proj.weight").view(12, 64, 768))
                + p("attn.c_proj.bias"))
        x = x + att
        m = gelu(layer_norm(x, p("ln_2.weight"), p("ln_2.bias")) @ p("mlp.c_fc.weight") + p("mlp.c_fc.bias"))
        x = x + H(f"mlp_out.{l}", m @ p("mlp.c_proj.weight") + p("mlp.c_proj.bias"))
    x = H("resid_post.11", x)                                             # the final running total
    if last_only:
        x = x[:, -1:]
    mu = x.mean(-1, keepdim=True)
    scale = H("scale", ((x - mu) ** 2).mean(-1, keepdim=True).add(1e-5).sqrt())   # the final LayerNorm's spread
    final = (x - mu) / scale * SD["ln_f.weight"] + SD["ln_f.bias"]
    return final @ WTE.T, cache

def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    return (x - mu) / torch.sqrt(((x - mu) ** 2).mean(-1, keepdim=True) + eps) * g + b

def gelu(x):                                   # GPT-2's smooth gate (the tanh form)
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

BOS = 50256                                         # <|endoftext|>, which the lecture put first, as TransformerLens does
def toks(text):
    return [BOS] + encode(text)

def word(i):
    return decode([i])

def tok1(w):
    """The single token for " w", or None if it takes more than one."""
    t = encode(" " + w)
    return t[0] if len(t) == 1 else None

def hn(l, h):
    return f"{l + 1}.{h + 1}"                       # the slides' head names count from 1

## Part 2 · Two sentences, one number (Plates I, III)

The *clean* sentence has John as the giver, so the answer is Mary. The *corrupted* one swaps the giver to Mary, so the answer flips to John. Everything else is identical, so whatever differs inside the model is about the giver.

One number to watch: the **logit difference**, Mary's score minus John's at the last word.

In [ ]:
CLEAN = "When Mary and John went to the store, John gave a drink to"
CORRUPT = "When Mary and John went to the store, Mary gave a drink to"
c_ids, k_ids = torch.tensor([toks(CLEAN)], device=DEV), torch.tensor([toks(CORRUPT)], device=DEV)
MARY, JOHN = tok1("Mary"), tok1("John")
print("tokens:", [word(i) for i in c_ids[0]])
check("the clean sentence's tokens", [word(i) for i in c_ids[0].tolist()], LECTURE["tokens"])

lc, cc = gpt2(c_ids, keep="all")
lk, ck = gpt2(k_ids, keep="all")
def top5(logits):
    p = logits.softmax(-1)
    v, i = p.topk(5)
    return [[word(int(a)), round(float(b), 4)] for a, b in zip(i, v)]
for name, lg in (("clean", lc), ("corrupted", lk)):
    print(f"{name:9s} next word:", "  ".join(f"{w!r} {p:.1%}" for w, p in top5(lg[0, -1])))
check("clean: the top next word", top5(lc[0, -1])[0][0], LECTURE["top5_clean"][0][0])
check("clean: its chance", top5(lc[0, -1])[0][1], LECTURE["top5_clean"][0][1], 0.0002)
check("corrupted: the top next word", top5(lk[0, -1])[0][0], LECTURE["top5_corrupt"][0][0])
check("corrupted: its chance", top5(lk[0, -1])[0][1], LECTURE["top5_corrupt"][0][1], 0.0002)

LD = lambda lg: float(lg[0, -1, MARY] - lg[0, -1, JOHN])
ld_c, ld_k = LD(lc), LD(lk)
print(f"logit difference, Mary minus John: clean {ld_c:+.2f}, corrupted {ld_k:+.2f}")
check("logit difference, clean", round(ld_c, 4), LECTURE["ld_clean"], 0.002)
check("logit difference, corrupted", round(ld_k, 4), LECTURE["ld_corrupt"], 0.002)
frac = lambda ld: (ld - ld_k) / (ld_c - ld_k)       # 0 = as corrupted, 1 = fully restored

## Part 3 · The 195 transplants (Plates IV–VIII)

A transplant: run the corrupted sentence, but at one place (one token, one depth) replace its running total with the clean run's, and see how much of the clean answer comes back. 15 tokens × 13 depths (before block 1 … after block 12) = 195 places.

In [ ]:
DEPTHS = [f"resid_pre.{l}" for l in range(12)] + ["resid_post.11"]
T = c_ids.shape[1]
grid = np.zeros((13, T))
for di, name in enumerate(DEPTHS):
    for pos in range(T):
        def put(x, pos=pos, name=name):
            x = x.clone(); x[:, pos] = cc[name][:, pos]; return x
        grid[di, pos] = frac(LD(gpt2(k_ids, hooks={name: put})[0]))
print("restored, at the second John (column 10):", np.round(grid[:, 10], 2))
print("restored, at the last word 'to' (column 14):", np.round(grid[:, 14], 2))
for d, name in ((0, "before block 1"), (8, "after block 8"), (9, "after block 9")):
    check(f"transplant at the second John, {name}", round(grid[d, 10], 4), LECTURE["john_col"][d], 0.002)
for d, name in ((8, "after block 8"), (9, "after block 9")):
    check(f"transplant at the last word, {name}", round(grid[d, 14], 4), LECTURE["to_col"][d], 0.002)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.imshow(grid, cmap="Greys", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(T)); ax.set_xticklabels([word(i) for i in c_ids[0].tolist()], rotation=60, fontsize=8)
ax.set_yticks(range(13)); ax.set_yticklabels(["before 1"] + [f"after {l}" for l in range(1, 13)], fontsize=8)
ax.set_title("share of the clean answer restored (dark = all)", fontsize=10)
plt.show()

**The last word doesn't know yet (Plate VII).** How far apart are the two runs' running totals at "to"? (TransformerLens, which the lecture used, removes each running total's average across its 768 numbers, which LayerNorm ignores anyway; we do the same before measuring.)

In [ ]:
centre = lambda v: v - v.mean(-1, keepdim=True)
dist = [float(centre(cc[n][0, -1] - ck[n][0, -1]).norm()) for n in DEPTHS]
print("distance between the runs at 'to', by depth:", np.round(dist, 1))
for d in (6, 8, 10):
    check(f"distance at 'to' after block {d}", round(dist[d], 3), LECTURE["dist_end"][d], 0.01)

**Two directions (Plate VIII).** The reverse transplant: run the *clean* sentence and drop in the *corrupted* run's numbers. Where it breaks the answer is where the answer depends on that place.

In [ ]:
noise_grid = np.zeros((13, T))
for di, name in enumerate(DEPTHS):
    for pos in range(T):
        def put(x, pos=pos, name=name):
            x = x.clone(); x[:, pos] = ck[name][:, pos]; return x
        noise_grid[di, pos] = frac(LD(gpt2(c_ids, hooks={name: put})[0]))
check("reverse transplant at 'to', after block 9", round(noise_grid[9, 14], 4), LECTURE["noise_to_9"], 0.002)
check("reverse transplant at the second John, after block 8", round(noise_grid[8, 10], 4), LECTURE["noise_john_8"], 0.002)

## Part 4 · 144 heads, one at a time (Plate IX)

Now transplant one attention head's output at a time (its `z`, at every position), clean into corrupted.

In [ ]:
heads = np.zeros((12, 12))
for l in range(12):
    for h in range(12):
        def put(z, h=h, l=l):
            z = z.clone(); z[:, :, h] = cc[f"z.{l}"][:, :, h]; return z
        heads[l, h] = frac(LD(gpt2(k_ids, hooks={f"z.{l}": put})[0]))
order = np.argsort(heads.ravel())
print("most restoring:", ", ".join(f"{hn(*divmod(i, 12))} {heads.ravel()[i]:+.0%}" for i in order[::-1][:4]))
print("most harmful:  ", ", ".join(f"{hn(*divmod(i, 12))} {heads.ravel()[i]:+.0%}" for i in order[:2]))
check("head 9.11, one pair", round(heads[8, 10], 4), LECTURE["head_pair_9_11"], 0.002)
check("head 10.10, one pair", round(heads[9, 9], 4), LECTURE["head_pair_10_10"], 0.002)
check("head 11.8, one pair", round(heads[10, 7], 4), LECTURE["head_pair_11_8"], 0.002)

**192 prompts.** One pair could be a fluke. The lecture's dataset: 8 sentence templates × 24 prompts, with names, places and objects drawn at random (the same seed gives the same prompts). Each is corrupted the "ABC" way: all three name slots get fresh names, so nothing is repeated. Now we *break* the clean run: each head in turn gets its output from the ABC run, and we measure the relative change in the logit difference.

In [ ]:
import random
TEMPLATES = [
    "When [A] and [B] went to the [PLACE], [B] gave a [OBJECT] to",
    "When [B] and [A] went to the [PLACE], [B] gave a [OBJECT] to",
    "After [A] and [B] went to the [PLACE], [B] gave a [OBJECT] to",
    "Then, [B] and [A] went to the [PLACE]. [B] gave a [OBJECT] to",
    "While [A] and [B] were working at the [PLACE], [B] gave a [OBJECT] to",
    "Friends [A] and [B] found a [OBJECT] at the [PLACE]. [B] gave it to",
    "When [B] and [A] got a [OBJECT] at the [PLACE], [B] decided to give it to",
    "Then, [A] and [B] had a long argument, and afterwards [B] said to",
]
NAMES = ["Mary", "John", "Tom", "James", "Dan", "Sarah", "Michael", "David", "Paul", "Mark", "Emma", "Anna",
         "Kate", "Alice", "Bob", "Lisa", "Laura", "Jessica", "Daniel", "Chris", "Peter", "Robert", "Rachel",
         "Amy", "Steve", "Mike", "Andrew", "George", "Eric", "Ryan", "Sam", "Jack", "Jane", "Ben", "Adam", "Joe"]
PLACES = ["store", "park", "school", "hospital", "office", "restaurant", "house", "station", "beach", "garden"]
OBJECTS = ["drink", "ring", "book", "letter", "gift", "bone", "ball", "computer", "necklace", "kiss"]

rng = random.Random(0)
names = [n for n in NAMES if tok1(n) is not None]
places = [w for w in PLACES if tok1(w) is not None]
objects = [w for w in OBJECTS if tok1(w) is not None]
batches = []
for tpl in TEMPLATES:
    rows = []
    for _ in range(24):
        A, B_, C_, D_, E_ = rng.sample(names, 5)
        pl, ob = rng.choice(places), rng.choice(objects)
        fill = lambda a, b: tpl.replace("[A]", a).replace("[B]", b).replace("[PLACE]", pl).replace("[OBJECT]", ob)
        abc = tpl.replace("[PLACE]", pl).replace("[OBJECT]", ob)
        first = "[A]" if abc.index("[A]") < abc.index("[B]") else "[B]"
        second = "[B]" if first == "[A]" else "[A]"
        abc = abc.replace(first, C_, 1).replace(second, D_, 1).replace("[B]", E_, 1)
        rows.append((fill(A, B_), abc, tok1(A), tok1(B_), A, B_))
    c = torch.tensor([toks(r[0]) for r in rows], device=DEV)
    a = torch.tensor([toks(r[1]) for r in rows], device=DEV)
    t0 = toks(rows[0][0])
    pos = {"IO": t0.index(rows[0][2]), "S1": t0.index(rows[0][3]),
           "S2": len(t0) - 1 - t0[::-1].index(rows[0][3]), "END": len(t0) - 1}
    lc_, cc_ = gpt2(c, keep="all")
    la_, ca_ = gpt2(a, keep="all")
    batches.append(dict(c=c, io=torch.tensor([r[2] for r in rows], device=DEV), s=torch.tensor([r[3] for r in rows], device=DEV),
                        pos=pos, lc=lc_, cc=cc_, la=la_, ca=ca_))
print("one prompt:", rows[0][0], "| its ABC version:", rows[0][1])

def ld_b(b, logits):
    e = logits[:, b["pos"]["END"]]
    ar = torch.arange(len(b["io"]), device=DEV)
    return e[ar, b["io"]] - e[ar, b["s"]]
base = torch.cat([ld_b(b, b["lc"]) for b in batches])
top1 = np.mean([float((b["lc"][:, b["pos"]["END"]].argmax(-1) == b["io"]).float().mean()) for b in batches])
print(f"{len(base)} prompts: mean logit difference {base.mean():.2f} clean, "
      f"{torch.cat([ld_b(b, b['la']) for b in batches]).mean():.2f} ABC; the answer is the top word in {top1:.1%}")
check("prompts", len(base), LECTURE["n_prompts"])
check("mean logit difference, clean", round(float(base.mean()), 4), LECTURE["ld_mean_clean"], 0.002)
check("mean logit difference, ABC", round(float(torch.cat([ld_b(b, b["la"]) for b in batches]).mean()), 4), LECTURE["ld_mean_abc"], 0.002)
check("the answer is the top word", round(top1, 4), LECTURE["top1_io"], 0.001)

def effect(make_hooks):
    vals = torch.cat([ld_b(b, gpt2(b["c"], hooks=make_hooks(b))[0]) for b in batches])
    return float(((vals - base) / base).mean())

def from_abc(heads_):
    def mk(b):
        hk = {}
        for l in sorted({l for l, _ in heads_}):
            idx = [h for ll, h in heads_ if ll == l]
            def fn(z, l=l, idx=idx, b=b):
                z = z.clone(); z[:, :, idx] = b["ca"][f"z.{l}"][:, :, idx]; return z
            hk[f"z.{l}"] = fn
        return hk
    return mk

t = time.time()
head192 = np.array([[effect(from_abc([(l, h)])) for h in range(12)] for l in range(12)])
worst = np.unravel_index(head192.argmin(), head192.shape)
print(f"({time.time() - t:.0f} s) the head whose swap hurts most: {hn(*worst)}, {head192[worst]:+.0%}")
check("head 12.3, swapped from the ABC run, 192 prompts", round(head192[11, 2], 4), LECTURE["head192_12_3"], 0.002)

## Part 5 · Who writes the answer, and what they look at (Plates X–XII, XIV)

**Direct push** (the logit lens on one head): take what a head writes at the last word, divide by the final LayerNorm's spread, and read it along the direction Mary − John (the answer's score minus the subject's). Before reading, remove its average across the 768 numbers, which LayerNorm would remove anyway. One more convention from TransformerLens, which the lecture used: each head's *value bias* (its own, plus what the LayerNorm's bias adds through the value weights) adds the same thing whatever the head looks at, so it is moved into a shared bias and left out of each head's push.

In [ ]:
def head_writes(z, l, e):
    """What each head of block l+1 writes at position e: [B, 12, 768], its value bias left out. That bias is the
    value weights' own bias plus what the LayerNorm's bias adds through them (TransformerLens folds the two together)."""
    p = lambda n: SD[f"h.{l}.{n}"]
    bV = (p("ln_1.bias") @ p("attn.c_attn.weight")[:, 1536:] + p("attn.c_attn.bias")[1536:]).view(12, 64)
    return torch.einsum("bhe,hed->bhd", z[:, e] - bV, SD[f"h.{l}.attn.c_proj.weight"].view(12, 64, 768))

dla = torch.zeros(12, 12, device=DEV)
for b in batches:
    e = b["pos"]["END"]
    scale = b["cc"]["scale"][:, e]                                        # [B, 1]
    direction = (WTE[b["io"]] - WTE[b["s"]]) * SD["ln_f.weight"]          # [B, 768]
    for l in range(12):
        out = head_writes(b["cc"][f"z.{l}"], l, e)                       # each head's write at END
        dla[l] += torch.einsum("bhd,bd->h", centre(out) / scale[:, None], direction) / len(b["io"])
dla = (dla / len(batches)).cpu().numpy()
o = np.argsort(dla.ravel())
print("pushes most toward the answer:", ", ".join(f"{hn(*divmod(i, 12))} {dla.ravel()[i]:+.2f}" for i in o[::-1][:3]))
print("pushes most against it:       ", ", ".join(f"{hn(*divmod(i, 12))} {dla.ravel()[i]:+.2f}" for i in o[:2]))
check("direct push of 10.10", round(float(dla[9, 9]), 4), LECTURE["dla_10_10"], 0.002)
check("direct push of 10.7", round(float(dla[9, 6]), 4), LECTURE["dla_10_7"], 0.002)
check("direct push of 11.8", round(float(dla[10, 7]), 4), LECTURE["dla_11_8"], 0.002)
NM = [divmod(int(i), 12) for i in o[::-1][:3]]                             # the three name movers
check("the three name movers (code numbering)", [f"{l}.{h}" for l, h in NM], LECTURE["name_movers"])

In [ ]:
att = {k: torch.zeros(12, 12, device=DEV) for k in ("END>IO", "END>S2", "S2>S1", "S2>S1+1")}
for b in batches:
    p = b["pos"]
    for l in range(12):
        A = b["cc"][f"pattern.{l}"]                                       # [B, head, from, to]
        att["END>IO"][l] += A[:, :, p["END"], p["IO"]].mean(0)
        att["END>S2"][l] += A[:, :, p["END"], p["S2"]].mean(0)
        att["S2>S1"][l] += A[:, :, p["S2"], p["S1"]].mean(0)
        att["S2>S1+1"][l] += A[:, :, p["S2"], p["S1"] + 1].mean(0)
att = {k: (v / len(batches)).cpu().numpy() for k, v in att.items()}
check("10.10, from the last word to the answer's name", round(float(att["END>IO"][9, 9]), 4), LECTURE["att_end_io_10_10"], 0.001)
check("9.7, from the last word to the repeated name", round(float(att["END>S2"][8, 6]), 4), LECTURE["att_end_s2_9_7"], 0.001)
check("4.1, from the repeated name to its first copy", round(float(att["S2>S1"][3, 0]), 4), LECTURE["att_s2_s1_4_1"], 0.001)
check("6.6, from the repeated name to the word after its first copy", round(float(att["S2>S1+1"][5, 5]), 4), LECTURE["att_induction_6_6"], 0.001)

fig, axs = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, (k, title) in zip(axs, (("END>IO", "last word → the answer's name"), ("END>S2", "last word → the repeated name"))):
    ax.imshow(att[k], cmap="Greys", vmin=0, vmax=1)
    ax.set_xlabel("head"); ax.set_ylabel("block"); ax.set_title(title, fontsize=10)
    ax.set_xticks(range(0, 12, 3)); ax.set_xticklabels(range(1, 13, 3)); ax.set_yticks(range(0, 12, 3)); ax.set_yticklabels(range(1, 13, 3))
plt.show()

## Part 6 · A transplant along one wire (Plate XIII)

Path patching (Wang et al.): does head X matter *because of what it sends to* heads Y?
1. Run the clean prompt with X's output taken from the ABC run and every other head frozen to its clean output (MLPs recompute). Record what Y receives.
2. Run the clean prompt again, changing only Y's input (its query, or its value) to what it received in step 1, at one position.
The change in the logit difference is the effect along that one wire.

In [ ]:
def path_patch(sender, receivers, recv_pos):
    """receivers: list of (layer, head, 'q' | 'v'). Returns the mean relative change in the logit difference."""
    sl, sh = sender
    vals = []
    for b in batches:
        def freeze(z, l):
            z = b["cc"][f"z.{l}"].clone()
            if l == sl:
                z[:, :, sh] = b["ca"][f"z.{l}"][:, :, sh]
            return z
        hooks = {f"z.{l}": (lambda z, l=l: freeze(z, l)) for l in range(12)}
        keepers = sorted({f"{t}.{l}" for l, _, t in receivers})
        _, got = gpt2(b["c"], hooks=hooks, keep=keepers)
        p = b["pos"][recv_pos]
        hooks2 = {}
        for name in keepers:
            t, l = name.split(".")
            idx = [h for ll, h, tt in receivers if ll == int(l) and tt == t]
            def put(x, name=name, idx=idx):
                x = x.clone(); x[:, p, idx] = got[name][:, p, idx]; return x
            hooks2[name] = put
        vals.append(ld_b(b, gpt2(b["c"], hooks=hooks2)[0]))
    v = torch.cat(vals)
    return round(float(((v - base) / base).mean()), 4)

t = time.time()
recv = [(l, h, "q") for l, h in NM]
first_nm = min(l for l, _ in NM)
pq = {(l, h): path_patch((l, h), recv, "END") for l in range(first_nm) for h in range(12)}
SI = [k for k, _ in sorted(pq.items(), key=lambda kv: kv[1])[:4]]
print(f"({time.time() - t:.0f} s) into the name movers' queries: " + ", ".join(f"{hn(*k)} {pq[k]:+.1%}" for k in SI))
check("into the name movers' queries, from 9.7", pq[(8, 6)], LECTURE["path_q_9_7"], 0.0005)
check("the four heads found (code numbering)", [f"{l}.{h}" for l, h in SI], LECTURE["s_inhibition"])

recv2 = [(l, h, "v") for l, h in SI]
pv = {(l, h): path_patch((l, h), recv2, "S2") for l in range(min(l for l, _ in SI)) for h in range(12)}
top_v = sorted(pv.items(), key=lambda kv: kv[1])[:3]
print("into those heads' values at the repeated name: " + ", ".join(f"{hn(*k)} {v:+.1%}" for k, v in top_v))
check("into their values, from 6.6", pv[(5, 5)], LECTURE["path_v_6_6"], 0.0005)

## Part 7 · Self-repair (Plate XVI)

Knock out the three name movers (their outputs from the ABC run, everywhere). Their direct pushes add up to more than the whole logit difference, so the answer should collapse. Does it?

In [ ]:
ko = from_abc(NM)
after = torch.cat([ld_b(b, gpt2(b["c"], hooks=ko(b))[0]) for b in batches])
direct = float(sum(dla[l, h] for l, h in NM))
print(f"the name movers push {direct:.2f} directly; knocked out, the logit difference goes {base.mean():.2f} -> {after.mean():.2f}")
check("the name movers' direct push, together", round(direct, 4), LECTURE["ko_direct"], 0.003)
check("logit difference with the name movers knocked out", round(float(after.mean()), 4), LECTURE["ko_after"], 0.002)

dla_ko = torch.zeros(12, 12, device=DEV)
for b in batches:
    e = b["pos"]["END"]
    _, kc = gpt2(b["c"], hooks=ko(b), keep="all")
    scale = kc["scale"][:, e]
    direction = (WTE[b["io"]] - WTE[b["s"]]) * SD["ln_f.weight"]
    for l in range(12):
        out = head_writes(kc[f"z.{l}"], l, e)
        dla_ko[l] += torch.einsum("bhd,bd->h", centre(out) / scale[:, None], direction) / len(b["io"])
change = (dla_ko / len(batches)).cpu().numpy() - dla
print("heads that step in:", ", ".join(f"{hn(*divmod(i, 12))} {change.ravel()[i]:+.2f}" for i in np.argsort(-change.ravel())[:4]))
check("11.8 relaxes", round(float(change[10, 7]), 4), LECTURE["ko_change_11_8"], 0.003)

## Part 8 · Where a fact lives (Plates XVII–XX)

**Causal tracing** (Meng et al. 2022). "Steve Jobs was the co-founder of" → Apple. Blur the subject: add noise of three times the spread of subject-token arrows to the subject's token arrows. Then restore one clean number set at a time (a running total, or a window of three blocks' MLP or attention outputs) and see how much of P(Apple) comes back, averaged over 10 noise draws. The lecture drew its noise on the CPU from seed 0, so we do too, and get the same draws.

In [ ]:
FACTS = [
    ("Eiffel Tower", "The Eiffel Tower is located in the city of", " Paris"), ("Colosseum", "The Colosseum is located in the city of", " Rome"),
    ("Space Needle", "The Space Needle is located in the city of", " Seattle"), ("Brandenburg Gate", "The Brandenburg Gate is located in the city of", " Berlin"),
    ("Big Ben", "Big Ben is located in the city of", " London"), ("Kremlin", "The Kremlin is located in the city of", " Moscow"),
    ("Sydney Opera House", "The Sydney Opera House is located in the city of", " Sydney"), ("Louvre", "The Louvre is located in the city of", " Paris"),
    ("Parthenon", "The Parthenon is located in the city of", " Athens"), ("Taj Mahal", "The Taj Mahal is located in the city of", " Agra"),
    ("Golden Gate Bridge", "The Golden Gate Bridge is located in the city of", " San"), ("Statue of Liberty", "The Statue of Liberty is located in the city of", " New"),
    ("Hollywood sign", "The Hollywood sign is located in the city of", " Los"), ("Michael Jordan", "Michael Jordan plays the sport of", " basketball"),
    ("Tiger Woods", "Tiger Woods plays the sport of", " golf"), ("Serena Williams", "Serena Williams plays the sport of", " tennis"),
    ("Wayne Gretzky", "Wayne Gretzky plays the sport of", " hockey"), ("Tom Brady", "Tom Brady plays the sport of", " football"),
    ("Babe Ruth", "Babe Ruth played the sport of", " baseball"), ("iPhone", "The iPhone is made by", " Apple"),
    ("Prius", "The Toyota Prius is made by", " Toyota"), ("Windows", "Windows was developed by", " Microsoft"),
    ("PlayStation", "The PlayStation is made by", " Sony"), ("Mount Fuji", "Mount Fuji is located in the country of", " Japan"),
    ("Great Wall", "The Great Wall is located in the country of", " China"), ("Nile", "The Nile flows through the country of", " Egypt"),
    ("Toyota", "Toyota is headquartered in the country of", " Japan"), ("BMW", "BMW is headquartered in the country of", " Germany"),
    ("Nokia", "Nokia is headquartered in the country of", " Finland"), ("Danielle Darrieux", "The mother tongue of Danielle Darrieux is", " French"),
    ("Vladimir Putin", "The mother tongue of Vladimir Putin is", " Russian"), ("Shakespeare", "William Shakespeare was born in the country of", " England"),
    ("Picasso", "Pablo Picasso was born in the country of", " Spain"), ("Einstein", "Albert Einstein was born in the country of", " Germany"),
    ("Mozart", "Wolfgang Amadeus Mozart was born in the country of", " Austria"), ("Brazil", "The official language of Brazil is", " Portuguese"),
    ("Mexico", "The official language of Mexico is", " Spanish"), ("Steve Jobs", "Steve Jobs was the co-founder of", " Apple"),
    ("Bill Gates", "Bill Gates was the co-founder of", " Microsoft"), ("Larry Page", "Larry Page was the co-founder of", " Google"),
    ("Mark Zuckerberg", "Mark Zuckerberg is the founder of", " Facebook"), ("Harry Potter", "Harry Potter was written by", " J"),
    ("Hamlet", "Hamlet was written by", " William"), ("Mona Lisa", "The Mona Lisa was painted by", " Leonardo"),
    ("Starry Night", "The Starry Night was painted by", " Vincent"), ("Beatles", "The Beatles were a band from the city of", " Liverpool"),
]

def subject_span(prompt, subj):
    """Token positions (BOS = 0) whose characters overlap the subject."""
    ids, out, at = encode(prompt), [], 0
    st, en = prompt.index(subj), prompt.index(subj) + len(subj)
    for i, t in enumerate(ids):
        n = len(decode([t]))
        if at < en and at + n > st and n > 0:
            out.append(i + 1)
        at += n
    return out

known = []
for subj, prompt, ans in FACTS:
    ids = torch.tensor([toks(prompt)], device=DEV)
    a = encode(ans)[0]
    p = gpt2(ids)[0][0, -1].softmax(-1)
    if int((p > p[a]).sum()) == 0:
        known.append(dict(subject=subj, prompt=prompt, a=a, ids=ids, span=subject_span(prompt, subj), p=float(p[a])))
SIGMA = float(centre(WTE[torch.cat([k["ids"][0, k["span"]] for k in known])]).std())
print(f"GPT-2 small knows {len(known)} of {len(FACTS)} facts; the subject arrows' spread is {SIGMA:.4f}")
check("facts it knows", len(known), LECTURE["n_known"])
check("spread of the subject-token arrows", round(SIGMA, 4), LECTURE["sigma"], 0.0002)

In [ ]:
def trace(ids, span, a, sigma, gen, n_noise=10, window=3):
    """P(answer) with the subject blurred, then with one clean state restored at each (kind, block, position)."""
    T = ids.shape[1]
    noise = (torch.randn(n_noise, len(span), 768, generator=gen) * 3 * sigma).to(DEV)
    _, clean = gpt2(ids, keep="all")
    conds = [("none", -1, -1)] + [(kd, l, i) for kd in ("resid", "mlp", "attn") for l in range(12) for i in range(T)]
    probs = np.zeros(len(conds))
    for s in range(0, len(conds), 64):
        cs = conds[s:s + 64]
        n = len(cs)
        nz = noise.repeat(n, 1, 1)
        def corrupt(e, nz=nz):
            e = e.clone(); e[:, span] = e[:, span] + nz; return e
        hooks = {"embed": corrupt}
        for kd, hk in (("resid", "resid_pre"), ("mlp", "mlp_out"), ("attn", "attn_out")):
            for l in range(12):
                sel = [(j, ci) for j, (ck, cl, ci) in enumerate(cs) if ck == kd and
                       ((cl == l) if kd == "resid" else (max(0, cl - window // 2) <= l <= min(11, cl + window // 2)))]
                if not sel:
                    continue
                rows = torch.tensor([j * n_noise + r for j, _ in sel for r in range(n_noise)], device=DEV)
                cols = torch.tensor([ci for _, ci in sel for r in range(n_noise)], device=DEV)
                # "resid" restores the running total after block l+1, which is the input to block l+2
                name = f"{hk}.{l + 1}" if kd == "resid" and l < 11 else ("resid_post.11" if kd == "resid" else f"{hk}.{l}")
                def restore(x, rows=rows, cols=cols, name=name):
                    x = x.clone(); x[rows, cols] = clean[name][0, cols]; return x
                hooks[name] = restore
        lg = gpt2(ids.repeat(n * n_noise, 1), hooks=hooks, last_only=True)[0][:, -1].softmax(-1)[:, a]
        probs[s:s + n] = lg.reshape(n, n_noise).mean(1).cpu().numpy()
    grids = {kd: probs[1 + m * 12 * T: 1 + (m + 1) * 12 * T].reshape(12, T) for m, kd in enumerate(("resid", "mlp", "attn"))}
    return float(probs[0]), grids

gen = torch.Generator().manual_seed(0)                 # on the CPU, as the lecture drew it
t = time.time()
for k in known:
    k["p_corrupt"], k["grids"] = trace(k["ids"], k["span"], k["a"], SIGMA, gen)
print(f"traced {len(known)} facts in {time.time() - t:.0f} s")
jobs = next(k for k in known if k["subject"] == "Steve Jobs")
print(f"Steve Jobs -> Apple: clean {jobs['p']:.1%}, blurred {jobs['p_corrupt']:.1%}; "
      f"the running total at ' Jobs' after block 4 restores {jobs['grids']['resid'][3, jobs['span'][-1]]:.1%}")
check("P(Apple), clean", round(jobs["p"], 4), LECTURE["jobs_top_clean"][1], 0.0005)
check("P(Apple) restored by the running total at ' Jobs' after block 4", round(float(jobs["grids"]["resid"][3, jobs["span"][-1]]), 4),
      LECTURE["jobs_resid_after_4"], 0.002)
check("mean P(answer), clean", round(float(np.mean([k["p"] for k in known])), 4), LECTURE["p_clean_mean"], 0.0005)
check("mean P(answer), blurred", round(float(np.mean([k["p_corrupt"] for k in known])), 4), LECTURE["p_corrupt_mean"], 0.002)

**Averaged over the 27 facts (Plate XIX).** Meng et al.'s summary groups positions by type (first, middle and last subject token, the first word after it, further words, the last word) and averages the *indirect effect*: the restored probability minus the blurred one.

In [ ]:
def by_token_type(facts):
    tot = {kd: np.zeros((6, 12)) for kd in ("resid", "mlp", "attn")}
    cnt = np.zeros(6)
    for f in facts:
        sp, T = f["span"], f["ids"].shape[1]
        def cat(i):
            if i == T - 1: return 5
            if i == sp[0]: return 0
            if i == sp[-1]: return 2
            if sp[0] < i < sp[-1]: return 1
            if i == sp[-1] + 1: return 3
            return 4 if i > sp[-1] else None
        for c in range(6):
            idx = [i for i in range(T) if cat(i) == c]
            if idx:
                cnt[c] += 1
                for kd in tot:
                    tot[kd][c] += (f["grids"][kd][:, idx] - f["p_corrupt"]).mean(1)
    return {kd: tot[kd] / np.maximum(cnt[:, None], 1) for kd in tot}
avg = by_token_type(known)
print("MLP at the last subject token, blocks 1-3:", np.round(avg["mlp"][2, :3], 3))
print("attention at the last word, blocks 9-11:   ", np.round(avg["attn"][5, 8:11], 3))
for l in range(2):
    check(f"MLP at the last subject token, block {l + 1}", round(float(avg["mlp"][2, l]), 4), LECTURE["mlp_last_subject"][l], 0.003)
check("attention at the last word, block 10", round(float(avg["attn"][5, 9]), 4), LECTURE["attn_last_token_9_10"], 0.003)

fig, axs = plt.subplots(1, 3, figsize=(9.5, 2.6), sharey=True)
types = ["first subj.", "middle subj.", "last subj.", "first after", "further", "last word"]
for ax, kd in zip(axs, ("resid", "mlp", "attn")):
    ax.imshow(avg[kd], cmap="Greys", vmin=0, aspect="auto")
    ax.set_title({"resid": "running total", "mlp": "MLP (window 3)", "attn": "attention (window 3)"}[kd], fontsize=10)
    ax.set_xticks(range(0, 12, 3)); ax.set_xticklabels(range(1, 13, 3)); ax.set_xlabel("block")
axs[0].set_yticks(range(6)); axs[0].set_yticklabels(types, fontsize=8)
plt.show()

**The Eiffel Tower (Plate XX).** The paper's own example. GPT-2 small barely knows it; on its best wording Paris only just comes first. (The lecture also traced it on GPT-2 XL, 1.5 billion weights and a 6 GB download, too large for this notebook: there the MLPs at the end of "Eiffel" around block 17 restore 75%. That number is the lecture's, not re-run here.)

In [ ]:
PARIS = encode(" Paris")[0]
WORDINGS = ["The Eiffel Tower is in", "The Eiffel Tower is located in", "The Eiffel Tower is located in the city of",
            "The Eiffel Tower is in the city of", "The Eiffel Tower stands in the city of",
            "The Eiffel Tower is a famous landmark in the city of", "The Eiffel Tower is in downtown",
            "The Eiffel Tower, in the heart of", "The Eiffel Tower is one of the most famous sights in"]
tried = []
for w in WORDINGS:
    p = gpt2(torch.tensor([toks(w)], device=DEV))[0][0, -1].softmax(-1)
    tried.append((w, float(p[PARIS]), int((p > p[PARIS]).sum())))
    print(f"P(Paris) {tried[-1][1]:6.1%}  rank {tried[-1][2] + 1:2d}   {w}")
best = max(tried, key=lambda t: (t[2] == 0, t[1]))
check("the wording on which Paris comes first", best[0], LECTURE["eiffel_best"])
check("P(Paris) on it", round(best[1], 4), LECTURE["eiffel_p"], 0.0005)
print(f"total run time: {(time.time() - T0_START) / 60:.1f} min")

## Your turn

1. **Transplant at "store".** Take the 195-cell map: what does a transplant at "store" restore at any depth? Why? *Hint: `grid[:, 8]`; does "store" differ between the two sentences?*
2. **Another pair.** Make your own clean/corrupted pair with different names (say Anna and Tom) and redo the transplant at the last word after blocks 8 and 9. *Hint: copy Part 2 with new sentences; `tok1()` checks a name is one token.*
3. **Knock out more.** Add the three heads that stepped in most (Part 7) to the knock-out. Does the logit difference finally collapse? *Hint: `from_abc(NM + extra)`.*
4. **Trace a fact of your own.** Pick a fact GPT-2 small knows from `known` and plot its running-total trace. *Hint: `k["grids"]["resid"]` is blocks × positions.*

In [ ]:
# 1. TODO: what does a transplant at "store" restore?

In [ ]:
# 2. TODO: your own pair

In [ ]:
# 3. TODO: knock out the name movers and the heads that stepped in

In [ ]:
# 4. TODO: trace a fact of your own

## Solutions

In [ ]:
# 1. "store" is the same word in both sentences, and nothing about the giver has reached it: restoring it changes nothing.
print("restored at 'store', every depth:", np.round(grid[:, 8], 3))

In [ ]:
# 2. Your own pair
c2, k2 = (torch.tensor([toks(s)], device=DEV) for s in ("When Anna and Tom went to the park, Tom gave a ball to",
                                                       "When Anna and Tom went to the park, Anna gave a ball to"))
A_, B_ = tok1("Anna"), tok1("Tom")
ld2 = lambda lg: float(lg[0, -1, A_] - lg[0, -1, B_])
l2c, c2c = gpt2(c2, keep="all"); l2k, _ = gpt2(k2)
for d in (8, 9):
    name = DEPTHS[d]
    def put(x, name=name):
        x = x.clone(); x[:, -1] = c2c[name][:, -1]; return x
    r = (ld2(gpt2(k2, hooks={name: put})[0]) - ld2(l2k)) / (ld2(l2c) - ld2(l2k))
    print(f"transplant at the last word after block {d}: {r:.0%} restored")

In [ ]:
# 3. Knock out more
extra = [divmod(int(i), 12) for i in np.argsort(-change.ravel())[:3]]
after2 = torch.cat([ld_b(b, gpt2(b["c"], hooks=from_abc(NM + extra)(b))[0]) for b in batches])
print(f"knocking out {', '.join(hn(*x) for x in NM + extra)}: logit difference {base.mean():.2f} -> {after2.mean():.2f}")

In [ ]:
# 4. Trace a fact of your own
k = next(f for f in known if f["subject"] == "Michael Jordan")
fig, ax = plt.subplots(figsize=(6, 3))
ax.imshow(k["grids"]["resid"], cmap="Greys", vmin=0, aspect="auto")
ax.set_xticks(range(k["ids"].shape[1])); ax.set_xticklabels([word(i) for i in k["ids"][0].tolist()], rotation=60, fontsize=8)
ax.set_ylabel("after block"); ax.set_yticks(range(0, 12, 3)); ax.set_yticklabels(range(1, 13, 3))
ax.set_title(f"{k['prompt']} -> P restored", fontsize=10)
plt.show()